In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, min, avg, count, round, first, when, unix_timestamp, explode, sum, desc, expr
import json

# Load configuration settings
with open('config.json', 'r') as file:
    config = json.load(file)

# Initialize the Spark Session
spark = SparkSession.builder \
    .appName("KitchenBatchAnalysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Define folder paths
kitchen_path = config.get("kitchen_data_path", "./output/kitchen_data")
dispatch_path = config.get("dispatch_data_path", "./output/dispatch_data")

print("Loading and cleaning saved data...")

try:
    # Read the files and drop rows with missing core IDs
    kitchen_df = spark.read.parquet(kitchen_path).dropna(subset=["batch_id", "recipe_id", "station_id"])
    dispatch_df = spark.read.parquet(dispatch_path).dropna(subset=["batch_id", "driver_id", "canteen_id"])

    # --- 1. Station Logic Check ---
    # Find any rows where the action does not match the station type
    wrong_stations = kitchen_df.filter(
        ((col("action") == "PREPARING") & (~col("station_id").startswith("prep"))) |
        ((col("action") == "COOKING") & (~col("station_id").startswith("cook"))) |
        ((col("action") == "PACKING") & (~col("station_id").startswith("pack")))
    )

    # --- 2. Kitchen Batch Analysis ---
    kitchen_summary = kitchen_df.groupby("batch_id").agg(
        first("recipe_id").alias("recipe"),
        round(max("expected_start_weight"), 2).alias("target_weight"),
        round(min("weight_kg"), 2).alias("lowest_weight_recorded"),
        round(max("temperature_celsius"), 1).alias("peak_temp"),
        max(when(col("weight_status") == "SUSPICIOUS_DROP", 1).otherwise(0)).alias("had_weight_issue"),
        min("event_timestamp").alias("kitchen_start"),
        max("event_timestamp").alias("kitchen_end")
    )

    kitchen_report = kitchen_summary.withColumn(
        "weight_loss_kg", 
        round(col("target_weight") - col("lowest_weight_recorded"), 2)
    ).withColumn(
        "kitchen_mins",
        round((unix_timestamp(col("kitchen_end"), "yyyy-MM-dd'T'HH:mm") - unix_timestamp(col("kitchen_start"), "yyyy-MM-dd'T'HH:mm")) / 60, 0).cast("int")
    )

    # --- 3. Dispatch Summary ---
    dispatch_summary = dispatch_df.groupby("batch_id").agg(
        first("canteen_id").alias("canteen"),
        first("driver_id").alias("driver"),
        min("event_timestamp").alias("dispatch_start"),
        max("event_timestamp").alias("delivery_time")
    )

    # --- 4. Quality Checks (System Alerts) ---
    print("\n--- System Alert: Wrong Station Assignments ---")
    if wrong_stations.count() == 0:
        print("All clear! All tasks happened at the correct stations.")
    else:
        wrong_stations.select("batch_id", "station_id", "action").show(truncate=False)

    print("\n--- System Alert: Failed Deliveries ---")
    failed_deliveries = kitchen_report.join(dispatch_summary, on="batch_id", how="left_anti")
    if failed_deliveries.count() == 0:
        print("All clear! No failed deliveries found.")
    else:
        failed_deliveries.select("batch_id", "recipe", "kitchen_end").show(truncate=False)

    print("\n--- System Alert: Missing Kitchen Data ---")
    crashed_kitchens = dispatch_summary.join(kitchen_report, on="batch_id", how="left_anti")
    if crashed_kitchens.count() == 0:
        print("All clear! No missing kitchen data found.")
    else:
        crashed_kitchens.select("batch_id", "canteen", "driver").show(truncate=False)

    # --- 5. Master End-to-End Report ---
    master_report = kitchen_report.join(dispatch_summary, on="batch_id", how="inner")

    master_report = master_report.withColumn(
        "delivery_mins",
        round((unix_timestamp(col("delivery_time"), "yyyy-MM-dd'T'HH:mm") - unix_timestamp(col("dispatch_start"), "yyyy-MM-dd'T'HH:mm")) / 60, 0).cast("int")
    ).withColumn(
        "total_process_mins",
        col("kitchen_mins") + col("delivery_mins")
    ).filter((col("kitchen_mins") >= 0) & (col("delivery_mins") >= 0))

    print("\n--- Master End-to-End Report ---")
    master_report.select("batch_id", "recipe", "canteen", "driver", "kitchen_mins", "delivery_mins", "total_process_mins").show(20, truncate=False)

    # Save the final joined report so the loader can find it
    master_report.write.mode("overwrite").parquet("./output/master_report_final")
    print("Master report saved to disk. Ready for DB loading.")

    # --- 6. Final Performance Reports ---
    print("\n--- Recipe Performance Summary ---")
    kitchen_report.groupby("recipe").agg(
        count("batch_id").alias("total_cooked"),
        round(avg("kitchen_mins"), 1).alias("avg_cook_time"),
        round(avg("weight_loss_kg"), 2).alias("avg_weight_loss")
    ).orderBy("avg_cook_time").show(truncate=False)

    print("\n--- Driver Performance Report ---")
    master_report.groupby("driver").agg(
        count("batch_id").alias("total_deliveries"),
        round(avg("delivery_mins"), 1).alias("avg_delivery_time"),
        expr("mode(canteen)").alias("most_visited_canteen")
    ).orderBy(desc("total_deliveries")).show(truncate=False)

except Exception as e:
    print(f"Error: Could not process data. Details: {e}")

spark.stop()

26/03/20 15:51:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Loading and cleaning saved data...

--- System Alert: Wrong Station Assignments ---
All clear! All tasks happened at the correct stations.

--- System Alert: Failed Deliveries ---
All clear! No failed deliveries found.

--- System Alert: Missing Kitchen Data ---
All clear! No missing kitchen data found.

--- Master End-to-End Report ---
+----------+------------+-------+------+------------+-------------+------------------+
|batch_id  |recipe      |canteen|driver|kitchen_mins|delivery_mins|total_process_mins|
+----------+------------+-------+------+------------+-------------+------------------+
|BATCH-1001|fish_curry  |cant_01|drv_02|60          |60           |120               |
|BATCH-1002|chicken_rice|cant_02|drv_03|69          |57           |126               |
|BATCH-1003|chicken_rice|cant_04|drv_01|48          |74           |122               |
|BATCH-1004|chicken_rice|cant_04|drv_04|51          |58           |109               |
|BATCH-1005|chicken_rice|cant_01|drv_05|62          